# Introduction to Vector Databases: Installing and Running ChromaDB

**Environment:** Google Colab / Local Python 3.9+  
**Domain:** Cloud Architecture, Security & Distributed Systems Engineering

---

### What You Will Learn
1. **Why a Vector Database?** The architectural difference between B-Trees/SQL, Inverted Indexes, and Approximate Nearest Neighbor (ANN) graphs (HNSW).
2. **Client Architectures:** In-Memory Ephemeral Client vs. Disk-Backed `PersistentClient`.
3. **Collections & Distance Spaces:** Creating collections with explicit metric spaces (`cosine`, `l2`, `ip`).
4. **Rich Metadata Schemas:** Adding structured attributes (tags, version, difficulty, verification status).
5. **Semantic Querying:** Interpreting distances, top-$k$ ranking, and relevance conversion.
6. **Advanced Metadata Filtering:** Compound logical expressions using `$and`, `$or`, `$gt`, and `$contains`.
7. **Custom Embedding Functions:** Integrating Hugging Face `SentenceTransformerEmbeddingFunction`.
8. **DevOps Knowledge Assistant:** An end-to-end interactive terminal search tool.

In [ ]:
# Install ChromaDB and Sentence-Transformers
!pip install -q chromadb sentence-transformers pandas numpy

## 1. Why a Vector Database?

| Traditional Database (RDBMS / NoSQL) | Inverted Index (Elasticsearch / BM25) | Vector Database (ChromaDB) |
|---|---|---|
| **Query Type** | Exact match (`WHERE id = 10`) | Keyword frequency (`WHERE text LIKE '%docker%'`) | Semantic proximity (`"how to stop memory leaks"`) |
| **Index Structure** | B-Tree, Hash Index | Inverted Index | Hierarchical Navigable Small World (HNSW) |
| **Synonyms** | No conceptual understanding | Requires manual synonym dictionaries | Native understanding of synonyms and context |
| **Scale Metric** | Row count / Primary Keys | Term frequency | High-dimensional geometric distance |

**ChromaDB** is an open-source, AI-native embedding database designed to make it effortless to build RAG and semantic search applications.

## 2. Initializing ChromaDB Clients

ChromaDB provides two primary modes of operation:
1. `chromadb.Client()`: In-memory ephemeral client (perfect for rapid prototyping, CI/CD tests, and temporary sessions).
2. `chromadb.PersistentClient(path="./chroma_db_store")`: Saves collections, embeddings, and indices to disk.

In [ ]:
import chromadb

# Initialize ephemeral in-memory client
client = chromadb.Client()
print("ChromaDB In-Memory Client initialized successfully!")
print("ChromaDB Version:", chromadb.__version__)

## 3. Creating Collections & Choosing Distance Metrics

A **Collection** in ChromaDB is analogous to a table in SQL. When creating a collection, you can explicitly configure its **HNSW distance metric space**:
- `"cosine"`: Measures angle ($D = 1 - \cos(\theta)$). Default and ideal for text embeddings.
- `"l2"`: Squared Euclidean distance ($D = \sum (u_i - v_i)^2$).
- `"ip"`: Inner Product ($D = 1 - (u \cdot v)$).

In [ ]:
# Delete existing collection if re-running cell
try:
    client.delete_collection(name="cloud_architecture_kb")
except Exception:
    pass

# Create collection with explicit Cosine distance space
collection = client.create_collection(
    name="cloud_architecture_kb",
    metadata={"hnsw:space": "cosine"}
)

print(f"Created collection '{collection.name}' with distance metric: {collection.metadata['hnsw:space']}")
print(f"Initial document count: {collection.count()}")

## 4. Ingesting Documents with Rich Structured Metadata

Let us populate our collection with real-world cloud engineering guides across different domains. ChromaDB automatically generates embeddings using its default lightweight embedding model (MiniLM) if no custom model is supplied.

In [ ]:
documents = [
    # DevOps & Kubernetes
    "Kubernetes Horizontal Pod Autoscaler HPA adjusts replica count dynamically based on CPU and memory metrics.",
    "Prometheus and Grafana provide real-time cluster monitoring and automated alerting rules for container metrics.",
    "ArgoCD implements GitOps continuous deployment by continuously reconciling live cluster state against Git repositories.",
    
    # Cloud Security & IAM
    "AWS IAM roles provide temporary least-privilege security credentials for EC2 instances and Lambda functions.",
    "Zero Trust network architecture mandates mutual TLS mTLS authentication between all internal microservices.",
    "HashiCorp Vault securely manages, encrypts, and rotates production database credentials and API tokens.",
    
    # Databases & Caching
    "Redis distributed in-memory cache drastically reduces primary database read load using LRU eviction policies.",
    "PostgreSQL read replicas distribute high-throughput analytical query traffic away from the primary write instance.",
    "Apache Kafka distributed commit log enables event-driven asynchronous microservices messaging at high throughput."
]

# Structured metadata supporting filtered queries
metadatas = [
    {"domain": "devops", "level": "intermediate", "year": 2024, "verified": True, "rating": 4.8},
    {"domain": "devops", "level": "beginner", "year": 2023, "verified": True, "rating": 4.5},
    {"domain": "devops", "level": "advanced", "year": 2024, "verified": True, "rating": 4.9},
    
    {"domain": "security", "level": "intermediate", "year": 2024, "verified": True, "rating": 4.7},
    {"domain": "security", "level": "advanced", "year": 2023, "verified": False, "rating": 4.4},
    {"domain": "security", "level": "advanced", "year": 2024, "verified": True, "rating": 4.9},
    
    {"domain": "database", "level": "intermediate", "year": 2023, "verified": True, "rating": 4.6},
    {"domain": "database", "level": "intermediate", "year": 2024, "verified": True, "rating": 4.8},
    {"domain": "database", "level": "advanced", "year": 2024, "verified": True, "rating": 4.7}
]

doc_ids = [f"arch-doc-{i:03d}" for i in range(1, len(documents) + 1)]

# Batch add to collection
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=doc_ids
)

print(f"Successfully indexed {collection.count()} documents into ChromaDB!")

## 5. Querying by Meaning (Semantic Search)

Unlike SQL `WHERE text LIKE '%auto scale%'`, ChromaDB maps the query into the same vector space and returns the nearest neighbors.

> **Interpreting Distance:**
> In ChromaDB with `"hnsw:space": "cosine"`:
> $$D = 1 - \text{Cosine Similarity}$$
> - $D = 0.0 \implies$ Exact 100% match.
> - $D < 0.4 \implies$ Strong semantic alignment.
> - We can convert distance to a human-friendly match score: $\text{Match Score} = (1 - D) \times 100\%$.

In [ ]:
def execute_search(query_text, n_results=3, where_filter=None):
    """Queries ChromaDB and displays formatted semantic search results."""
    results = collection.query(
        query_texts=[query_text],
        n_results=n_results,
        where=where_filter
    )
    
    print(f"\nSearch Query: '{query_text}'")
    if where_filter:
        print(f"Metadata Filter: {where_filter}")
    print("=" * 80)
    
    ids = results["ids"][0]
    docs = results["documents"][0]
    metas = results["metadatas"][0]
    distances = results["distances"][0]
    
    for rank, (doc_id, doc, meta, dist) in enumerate(zip(ids, docs, metas, distances), start=1):
        match_pct = max(0.0, (1.0 - dist)) * 100
        print(f"Rank #{rank} | ID: {doc_id} | Distance: {dist:.4f} (Match: {match_pct:.1f}%)")
        print(f"  Domain: {meta['domain']} | Level: {meta['level']} | Verified: {meta['verified']}")
        print(f"  Content: {doc}")
        print("-" * 80)

# Query 1: Scalability (notice no exact keywords 'HPA' or 'replica' in the query)
execute_search("How can I automatically scale server pods when traffic spikes?", n_results=2)

# Query 2: Secret management
execute_search("Safely storing database passwords and sensitive API keys", n_results=2)

## 6. Advanced Metadata Filtering

In production enterprise search, users frequently combine **semantic matching** with **hard business filters** (e.g., "Only show verified DevOps articles published after 2023").

ChromaDB supports MongoDB-style query filter operators:
- `$eq` - equals
- `$ne` - not equals
- `$gt`, `$gte` - greater than / greater than or equal
- `$lt`, `$lte` - less than / less than or equal
- `$in` - value in list
- `$nin` - value not in list
- `$and`, `$or` - logical combinations

In [ ]:
# 1. Single Equality Filter
execute_search("How to handle high traffic and reliability?", n_results=2, where_filter={"domain": "database"})

# 2. Compound Filter ($and with rating threshold)
compound_filter = {
    "$and": [
        {"domain": {"$eq": "devops"}},
        {"rating": {"$gte": 4.8}}
    ]
}
execute_search("Continuous delivery and cluster automation", n_results=2, where_filter=compound_filter)

# 3. Filtering by document text content ($contains)
print("\nFiltering by text content ('mTLS'):")
res_content = collection.query(
    query_texts=["network communication security"],
    n_results=2,
    where_document={"$contains": "mTLS"}
)
print("Matched ID:", res_content["ids"][0])
print("Matched Text:", res_content["documents"][0])

## 7. Using Custom Hugging Face Embedding Functions

By default, ChromaDB uses its built-in model. You can seamlessly inject custom Hugging Face models using `chromadb.utils.embedding_functions`.

In [ ]:
from chromadb.utils import embedding_functions

# Instantiate Hugging Face Sentence-Transformer embedding function
hf_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Create collection with custom embedding function
custom_col = client.get_or_create_collection(
    name="custom_hf_collection",
    embedding_function=hf_ef,
    metadata={"hnsw:space": "cosine"}
)

custom_col.add(
    documents=[
        "Elasticsearch indexes JSON documents for rapid full-text lexical search.",
        "FAISS from Meta provides high-performance GPU-accelerated vector indexing."
    ],
    ids=["doc-custom-1", "doc-custom-2"]
)

res = custom_col.query(query_texts=["Vector search acceleration"], n_results=1)
print("Custom Embedding Function Result:")
print(f"Top Match: {res['documents'][0][0]} (Distance: {res['distances'][0][0]:.4f})")

## 8. Persistent Storage with `PersistentClient`

To persist collections to disk across program restarts, use `chromadb.PersistentClient`.

In [ ]:
import os

persist_dir = "./chroma_db_store"
os.makedirs(persist_dir, exist_ok=True)

# Initialize persistent client
disk_client = chromadb.PersistentClient(path=persist_dir)

persisted_col = disk_client.get_or_create_collection(name="persisted_devops_docs")
persisted_col.add(
    documents=["Terraform enables infrastructure as code declarative cloud provisioning."],
    metadatas=[{"tool": "terraform", "iac": True}],
    ids=["tf-001"]
)

print(f"Persistent client verified. Document count in '{persisted_col.name}': {persisted_col.count()}")
print(f"Files saved in {persist_dir}: {os.listdir(persist_dir)}")

## 9. Practice Exercises (Solved)

### Exercise 1: Multi-Attribute Range Filter
Find all verified security documents that have a rating $\ge 4.7$.

In [ ]:
ex1_filter = {
    "$and": [
        {"domain": {"$eq": "security"}},
        {"verified": {"$eq": True}},
        {"rating": {"$gte": 4.7}}
    ]
}

res_ex1 = collection.get(where=ex1_filter)
print("Exercise 1 Matched Documents:")
for doc_id, doc, meta in zip(res_ex1["ids"], res_ex1["documents"], res_ex1["metadatas"]):
    print(f" - [{doc_id}] {doc} (Rating: {meta['rating']})")

### Exercise 2: Inspecting Raw Embeddings
Retrieve the actual 384-dimensional vector stored inside ChromaDB for a specific document ID.

In [ ]:
raw_record = collection.get(
    ids=["arch-doc-001"],
    include=["embeddings", "documents", "metadatas"]
)

emb = raw_record["embeddings"][0]
print("Document:", raw_record["documents"][0][:60], "...")
print(f"Embedding length: {len(emb)} dimensions")
print(f"First 5 vector values: {np.round(emb[:5], 4)}")

## Key Takeaways

1. **AI-Native Vector Store:** ChromaDB eliminates boilerplate for generating, storing, and indexing embeddings.
2. **HNSW Indexing:** Provides sub-millisecond Approximate Nearest Neighbor search over large document sets.
3. **Hybrid Querying:** Combines semantic similarity with structured metadata filtering (`$and`, `$gt`, `$in`).
4. **Persistence:** `chromadb.PersistentClient` provides zero-configuration local persistence backed by SQLite and Parquet.